# Pachinko simulator — notebook version

A Galton-board style pachinko machine: a coin dropped on the apex peg bounces **left or right with a 50/50 chance** at every peg, and after 10 bounces lands in one of 11 bins.

What the drawing shows:

- **Step through** a drop one bounce at a time (widget button, or re-run the manual cell below).
- **Percentages**: green arrows mark the two possible next vertices, each labelled **50%**; under each bin the theoretical landing share is printed.
- **Last vertex / last edge**: the red edge + red ring mark the move the coin made last (also spelled out in the title line). By default only that last edge is drawn; the **Full path** toggle (or `draw_board(..., full_path=True)`) shows the whole travelled path instead.
- **Heatmap**: the **Heatmap** toggle (or `draw_board(..., heatmap=True)`) overlays the paths of *all* coins dropped so far — every edge is coloured light→dark blue by how many coins travelled it, so the 50/50 traffic pattern builds up visibly.

Requirements: `matplotlib` (needed), `ipywidgets` (optional, for the button UI). There is also a no-install browser version: open `pachinko_browser.html`.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

from pachinko import PachinkoBoard
from pachinko_mpl import draw_board

board = PachinkoBoard(rows=10)
board.drop_many(200)   # pre-fill the histogram a little
fig = draw_board(board)

## Interactive UI (ipywidgets)

**Drop coin** starts a coin on the apex, **Next bounce** steps it down one 50/50 decision at a time, **Finish drop** runs it to the bottom, **Drop 100** fills the histogram instantly.

In [ ]:
try:
    import ipywidgets as W
    from IPython.display import display, clear_output

    ui_board = PachinkoBoard(rows=10)
    out = W.Output()
    show_pct = W.Checkbox(value=True, description="Show %", indent=False)
    full_path = W.Checkbox(value=False, description="Full path", indent=False)
    heatmap = W.Checkbox(value=False, description="Heatmap", indent=False)

    def refresh(*_):
        with out:
            clear_output(wait=True)
            fig = draw_board(ui_board, show_pct=show_pct.value,
                             full_path=full_path.value, heatmap=heatmap.value)
            display(fig)
            plt.close(fig)

    def on_drop(_):
        if ui_board.coin is not None and not ui_board.coin.landed:
            ui_board.finish()          # previous coin falls the rest of the way
        ui_board.drop()
        refresh()

    def on_step(_):
        ui_board.bounce()
        refresh()

    def on_finish(_):
        ui_board.finish()
        refresh()

    def on_many(_):
        ui_board.drop_many(100)
        refresh()

    def on_reset(_):
        ui_board.reset()
        refresh()

    row = []
    for label, cb in [("Drop coin", on_drop), ("Next bounce", on_step),
                      ("Finish drop", on_finish), ("Drop 100", on_many),
                      ("Reset", on_reset)]:
        b = W.Button(description=label)
        b.on_click(cb)
        row.append(b)
    for chk in (show_pct, full_path, heatmap):
        chk.observe(refresh, names="value")

    display(W.HBox(row + [show_pct, full_path, heatmap]), out)
    refresh()
except ImportError:
    print("ipywidgets is not installed - use the manual cells below instead.")
    print("(pip install ipywidgets)")

## Manual step-through (no widgets needed)

Run the next cell once to set up, then **re-run the cell after it (Ctrl+Enter) again and again** — each run is one bounce; after a landing it drops a fresh coin.

In [ ]:
demo = PachinkoBoard(rows=10, seed=11)
demo.drop_many(300)
demo.drop()
fig = draw_board(demo)

In [ ]:
# Re-run this cell (Ctrl+Enter) to advance one bounce at a time.
if demo.coin is None or demo.coin.landed:
    demo.drop()
else:
    demo.bounce()
fig = draw_board(demo)

## The maths

Every bounce is an independent 50/50 choice, so a coin that ends in bin $k$ made exactly $k$ right-bounces out of 10. The landing probability is binomial:

$$P(\text{bin } k) = \binom{10}{k} \left(\tfrac{1}{2}\right)^{10}$$

which is what `board.bin_probability(k)` returns and what the percentages under the bins show. Drop a few hundred coins and the histogram converges to this bell shape.